# Change Point Detection

In this notebook, I create standard error estimates using bootstrapping, and use the estimates to conduct change point detection analysis (CUSUM and EWMA-residual).

In [1]:
# Libraries

import os
import pandas as pd
import numpy as np

In [2]:
os.getcwd()

"/Users/jananidhileepan/Desktop/Don't. Even./University/Imperial College London/Year 2/Dissertation/GitHub/fairness_smoke_alarm/src"

In [ ]:
from fairness_metrics import (
    dict_references,
    get_bin_edges,
    dem_parity_total,
    eq_odds_total,
    calibration_total,
    unfairness_score
)

## 1. Standard Error Estimation using Bootstrapping

In [5]:
hmda_pred = pd.read_csv("../data/processed/hmda_with_predictions.csv", index_col=0)
hmda_2007_pred = hmda_pred[hmda_pred["year"] == 2007].copy()

In [7]:
bin_edges = get_bin_edges(hmda_2007_pred)

def check_categories_present(df, attribute):
    """Raise if a bootstrap resample dropped a category, which would silently
    shift the positional indexing in dem_parity_calc / eq_odds_calc."""
    expected = sorted(hmda_2007_pred[attribute].dropna().unique())
    present = sorted(df[attribute].dropna().unique())
    if present != expected:
        missing = set(expected) - set(present)
        raise ValueError(f"Resample missing categories in '{attribute}': {missing}")

def bootstrap_fairness_metrics(df, bin_edges, n_bootstrap=200, random_state=0):
    """
    Bootstrap DP/EO/CAL on df.
    Returns a DataFrame with one row per resample, columns DP/EO/CAL,
    preserving the within-resample pairing needed for composite sigma-hat.
    """
    n = len(df)
    rng = np.random.default_rng(random_state)
    records = []

    for i in range(n_bootstrap):
        # Draw n row-indices with replacement; every column travels together per row
        idx = rng.integers(0, n, size=n)
        resample = df.iloc[idx]

        # Guard against a resample silently dropping a subgroup category
        for attribute in ["race", "sex", "ethnicity"]:
            check_categories_present(resample, attribute)

        dp = dem_parity_total(resample)
        eo = eq_odds_total(resample)
        cal = calibration_total(resample, bin_edges)

        records.append({"DP": dp, "EO": eo, "CAL": cal})

    return pd.DataFrame(records)

# Run it
boot_df = bootstrap_fairness_metrics(hmda_2007_pred, bin_edges, n_bootstrap=200, random_state=0)
boot_df.head()

,DP,EO,CAL
0,0.114333,0.100333,0.125485
1,0.114667,0.100500,0.125657
2,0.117000,0.101167,0.126087
3,0.115000,0.100000,0.125336
4,0.114000,0.101500,0.125973


In [ ]:
def sigma_hat(boot_df, alpha, beta, gamma):
    composite = alpha * boot_df["DP"] + beta * boot_df["EO"] + gamma * boot_df["CAL"]
    return composite.std()

## 2. Change Point Detection

### A. Method 1: Cumulative Sum (CUSUM)

In [ ]:
def cusum()